In [ ]:
## Nikolay Vorontsov,
## Mushroom task
## Inference with fine-tuned model

In [1]:
!pip install transformers huggingface_hub pip torch jsonlines regex
# Install necessary dependencies
!pip install transformers peft accelerate huggingface_hub
!pip install -q trl xformers wandb datasets einops sentencepiece
!pip install -U datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
import jsonlines
import re
import torch
import json
from transformers import AutoTokenizer, AutoModelForTokenClassification

from huggingface_hub import login


from google.colab import userdata

HUGGING_API = userdata.get('HUGGINGFACE_READ_AND_WRITE')

In [3]:

# Login to Hugging Face
login(token=HUGGING_API)


In [4]:
tokenizer = AutoTokenizer.from_pretrained("nicksnlp/llama-7B-hallucination")
model = AutoModelForTokenClassification.from_pretrained("nicksnlp/llama-7B-hallucination")

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

LlamaForTokenClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4

In [5]:
import torch

def infer_with_model(input_text):
    # Tokenize input text with offset mapping and special tokens
    inputs = tokenizer(input_text,
                       return_tensors="pt",
                       padding=True,
                       truncation=True,
                       max_length=512,
                       return_offsets_mapping=True,
                       add_special_tokens=True
                       )

    # Move input tensors to model's device
    offset_mapping = inputs.pop("offset_mapping")[0].tolist()  # Extract character spans
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Send to correct device

    # Predict token labels
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Model outputs
    predicted_labels = torch.argmax(logits, dim=-1)[0].tolist()  # Convert to list

    # Get tokenized words
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

    # Align tokens with labels and offset mappings
    labeled_tokens = []
    hallucinated_words = []

    for token, label, (start, end) in zip(tokens, predicted_labels, offset_mapping):
        if start == 0 and end == 0:  # Skip special tokens (e.g., [CLS], [SEP])
            continue

        labeled_tokens.append((token, label, (start, end)))  # Add character positions

        if label == 1:
            hallucinated_words.append(input_text[start:end])  # Extract hallucinated word

    return hallucinated_words, labeled_tokens


In [23]:
import regex as re

def export_spans_using_offsets(tokens, full_text):
    # Step 1: Find the "@@" marker in full_text
    trim_marker = "<@@>"
    marker_pos = full_text.find(trim_marker)

    if marker_pos == -1:
        raise ValueError("Trim marker <@@> not found in full_text!")

    # Step 2: Determine the new start position
    answer_start = marker_pos + len(trim_marker)

    # Step 3: Trim text after "@@" marker
    trimmed_text = full_text[answer_start:]

    spans = []
    start = None

    for token, label, (char_start, char_end) in tokens:
        if char_start == 0 and char_end == 0:  # Skip special tokens
            continue

        if char_end <= answer_start:  # Ignore tokens before "@@"
            continue

        # Adjust character positions to be relative to the trimmed text
        adj_start = max(char_start - answer_start, 0)
        adj_end = char_end - answer_start

        if label == 1:
            if start is None:
                start = adj_start  # Start a new span
        else:
            if start is not None:
                spans.append((start, adj_start))  # Close span
                start = None

    if start is not None:
        spans.append((start, adj_end))  # Close last span

    # Step 4: Extract spans from the trimmed text
    extracted_texts = [trimmed_text[start:end] for start, end in spans]

    print("Original Full Text:#"+ full_text)
    print("Trimmed Text:#"+ trimmed_text)
    print("Adjusted Spans of Label 1:", spans)
    print("Extracted Answer Texts:", extracted_texts)

    return trimmed_text, spans, extracted_texts


In [24]:
# Example usage of the inference function
question = "Which municipalities does the Italian commune of Ponzone border?"
input_text = " YES, YES, << Ponza\n"
full_text = question+"<@@>"+input_text
hallucinated_words, labeled_tokens = infer_with_model(full_text)

# Print the list of hallucinated words
print("Hallucinated words:")
print(hallucinated_words)
print(full_text)
print(labeled_tokens)


Hallucinated words:
['ities', ' of', 'zone', '@@', ' <<', 'za']
Which municipalities does the Italian commune of Ponzone border?@@ YES, YES, << Ponza

[('▁Which', 0, (0, 5)), ('▁municipal', 0, (5, 15)), ('ities', 1, (15, 20)), ('▁does', 0, (20, 25)), ('▁the', 0, (25, 29)), ('▁Italian', 0, (29, 37)), ('▁commune', 0, (37, 45)), ('▁of', 1, (45, 48)), ('▁Pon', 0, (48, 52)), ('zone', 1, (52, 56)), ('▁border', 0, (56, 63)), ('?', 0, (63, 64)), ('@@', 1, (64, 66)), ('▁YES', 0, (66, 70)), (',', 0, (70, 71)), ('▁YES', 0, (71, 75)), (',', 0, (75, 76)), ('▁<<', 1, (76, 79)), ('▁Pon', 0, (79, 83)), ('za', 1, (83, 85)), ('<0x0A>', 0, (85, 86))]


In [25]:
t, ch, s = export_spans_using_offsets(labeled_tokens, full_text)
print(t)
print(ch)

Original Full Text:#Which municipalities does the Italian commune of Ponzone border?@@ YES, YES, << Ponza

Trimmed Text:# YES, YES, << Ponza

Adjusted Spans of Label 1: [(10, 13), (17, 19)]
Extracted Answer Texts: [' <<', 'za']
 YES, YES, << Ponza

[(10, 13), (17, 19)]


In [41]:
t = "Yes, all arachnids have antennas. However, not all of them are visible to the naked eye."
#87, 88
t[87:88]

'.'

In [42]:
def test_inferences(validation_file, output_file):
  with jsonlines.open(validation_file) as reader, jsonlines.open(output_file, 'w') as writer:
    for datapoint in reader:
            model_output_text = datapoint.get("model_output_text", "")
            question = datapoint.get("model_input", "")
            qa_pair = question+"<@@>"+model_output_text

            hallucinated_words, labeled_tokens = infer_with_model(qa_pair)

            _, hard_labels, _ = export_spans_using_offsets(labeled_tokens, qa_pair)

            datapoint["hard_labels"] = hard_labels
            #datapoint["id"] = datapoint["id"].replace('_unlabeled', '')

            writer.write(datapoint)

In [47]:
# "a" creates the file if it doesn't exist
output_file = "/content/mushroom.en-tst.v1.jsonl_llama_7b-hallucinations"
#output_file = "/content/mushroom.en-val.v2._llama_7b-hallucinations"

with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/mushroom.en-tst.v1.jsonl_llama_7b-hallucinations is created or already exists.


In [48]:

# Example usage:
validation_file = "/content/mushroom.en-tst.v1.jsonl" #"/content/mushroom.en-val.v2.unlabeled.jsonl" #
test_inferences(validation_file, output_file)
print(f"Processed data written to {output_file}")

Original Full Text:#Did Alberto Fouillioux ever play in a world cup championship?@@ No, Albero Foulois was not in any of the FIFA World Cup finals.

Trimmed Text:# No, Albero Foulois was not in any of the FIFA World Cup finals.

Adjusted Spans of Label 1: [(0, 3), (4, 30), (37, 41), (56, 63), (64, 65)]
Extracted Answer Texts: [' No', ' Albero Foulois was not in', ' the', ' finals', '\n']
Original Full Text:#In which city was David Sandberg born?@@David Sandburg was born in Stockholm, Sweden.
Trimmed Text:#David Sandburg was born in Stockholm, Sweden.
Adjusted Spans of Label 1: [(10, 23), (26, 44)]
Extracted Answer Texts: ['burg was born', ' Stockholm, Sweden']
Original Full Text:#When did Andrei Kravchuk begin working on The Italian?@@ Andreï Krivochkin started working in the Italian in 1995.

Trimmed Text:# Andreï Krivochkin started working in the Italian in 1995.

Adjusted Spans of Label 1: [(0, 7), (18, 26), (34, 52), (55, 57), (58, 59)]
Extracted Answer Texts: [' Andreï', ' started

In [49]:
# Clean_output

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')
        print(data_to_resave['id'])
        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels'] if label]
        print(soft_labels)


             # Save the datapoint to the JSONL file
        with open(f"{output_file}_no_extra_keys_soft_labels_prob1.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")

tst-en-1
[[0, 3], [4, 30], [37, 41], [56, 63], [64, 65]]
[{'start': 0, 'prob': 1.0, 'end': 3}, {'start': 4, 'prob': 1.0, 'end': 30}, {'start': 37, 'prob': 1.0, 'end': 41}, {'start': 56, 'prob': 1.0, 'end': 63}, {'start': 64, 'prob': 1.0, 'end': 65}]
tst-en-2
[[10, 23], [26, 44]]
[{'start': 10, 'prob': 1.0, 'end': 23}, {'start': 26, 'prob': 1.0, 'end': 44}]
tst-en-3
[[0, 7], [18, 26], [34, 52], [55, 57], [58, 59]]
[{'start': 0, 'prob': 1.0, 'end': 7}, {'start': 18, 'prob': 1.0, 'end': 26}, {'start': 34, 'prob': 1.0, 'end': 52}, {'start': 55, 'prob': 1.0, 'end': 57}, {'start': 58, 'prob': 1.0, 'end': 59}]
tst-en-4
[[0, 1], [8, 19], [26, 37], [49, 52]]
[{'start': 0, 'prob': 1.0, 'end': 1}, {'start': 8, 'prob': 1.0, 'end': 19}, {'start': 26, 'prob': 1.0, 'end': 37}, {'start': 49, 'prob': 1.0, 'end': 52}]
tst-en-5
[[4, 8], [38, 44], [64, 68], [71, 74], [115, 119], [126, 140], [169, 182], [185, 186]]
[{'start': 4, 'prob': 1.0, 'end': 8}, {'start': 38, 'prob': 1.0, 'end': 44}, {'start': 64, '